# Multinomial Logistic Regression

This notebook computes the requested feature set from the raw combined data and runs multinomial logistic regression with leave-one-feature-out ablation.

Features used:
- MER
- Levenshtein similarity
- METEOR
- Idea-unit coverage
- Sentence length
- Syllable coverage

In [11]:
%pip install python-Levenshtein nltk jiwer silabeador -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\khann\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Ordinal Regression Ablation

## What this notebook does

This notebook turns the raw Spanish EIT data into a set of explicit engineered features, then compares several ordinal and non-ordinal models under the same cross-validation split.

The feature families are intentionally mixed:
- lexical overlap and edit-style similarity
- rubric coverage from the idea-unit workbook
- response length and syllable coverage
- BETO embeddings as a higher-capacity neural comparison

The key question is not only which model scores best, but which one is stable, interpretable, and realistic to ship to other researchers.

In [12]:
import pandas as pd
import numpy as np
import re
import unicodedata
from jiwer import mer
import Levenshtein
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from silabeador import Syllabification
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

df = pd.read_excel('../data/combined.xlsx')
df_ideas_raw = pd.read_excel('../data/idea_units_spanish_autoEIT.xlsx')
df_ideas_raw.columns = df_ideas_raw.columns.str.lower()


def clean_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    value = str(value).strip()
    return value if value else None


def normalize_text(value):
    value = clean_text(value)
    if value is None:
        return ''
    value = unicodedata.normalize('NFKD', value.lower())
    value = ''.join(char for char in value if not unicodedata.combining(char))
    value = re.sub(r'[^a-z0-9ñ\s]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()


def levenshtein_similarity(ref, hyp):
    ref = clean_text(ref)
    hyp = clean_text(hyp)
    if ref is None or hyp is None:
        return np.nan
    distance = Levenshtein.distance(ref, hyp)
    return 1 - distance / max(len(ref), len(hyp), 1)


def meteor_safe(ref, hyp):
    ref = clean_text(ref)
    hyp = clean_text(hyp)
    if ref is None or hyp is None:
        return np.nan
    try:
        return meteor_score([word_tokenize(ref.lower())], word_tokenize(hyp.lower()))
    except Exception:
        return np.nan


def count_syllables(text):
    text = clean_text(text)
    if text is None:
        return np.nan
    total = 0
    for word in text.lower().split():
        try:
            total += len(Syllabification(word).syllables)
        except Exception:
            total += max(1, sum(1 for char in word if char in 'aeiouáéíóú'))
    return max(total, 1)


def syllable_coverage(ref, hyp):
    ref_count = count_syllables(ref)
    hyp_count = count_syllables(hyp)
    if pd.isna(ref_count) or pd.isna(hyp_count) or ref_count == 0:
        return np.nan
    return hyp_count / ref_count


def sentence_length(hyp):
    hyp = clean_text(hyp)
    if hyp is None:
        return np.nan
    return len(hyp.split())


def infer_stimulus_column(frame):
    for column in frame.columns:
        if column == 'stimulus' or 'stimulus' in column.lower():
            return column
    raise ValueError('Could not find a stimulus column in the idea-units file.')


def infer_idea_units_column(frame):
    preferred = [column for column in frame.columns if 'idea' in column.lower() or 'unit' in column.lower()]
    if preferred:
        return preferred[0]
    for column in frame.columns:
        if column != 'stimulus':
            return column
    raise ValueError('Could not find an idea-units column in the idea-units file.')


def parse_idea_units(value):
    value = clean_text(value)
    if value is None:
        return []
    parts = [part.strip() for part in re.split(r'\s*::\s*', value) if part.strip()]
    return parts


def idea_unit_coverage_from_excel(transcription, idea_units):
    transcription_norm = normalize_text(transcription)
    if not transcription_norm or not idea_units:
        return np.nan
    covered = 0
    for unit in idea_units:
        unit_norm = normalize_text(unit)
        if not unit_norm:
            continue
        pattern = r'(?<!\w)' + re.escape(unit_norm) + r'(?!\w)'
        if re.search(pattern, transcription_norm):
            covered += 1
    return covered / len(idea_units)


stimulus_col = infer_stimulus_column(df_ideas_raw)
idea_units_col = infer_idea_units_column(df_ideas_raw)

idea_units_df = df_ideas_raw[[stimulus_col, idea_units_col]].copy()
idea_units_df = idea_units_df.rename(columns={stimulus_col: 'stimulus', idea_units_col: 'idea_units_raw'})
idea_units_df['idea_units'] = idea_units_df['idea_units_raw'].apply(parse_idea_units)
idea_units_df = idea_units_df[['stimulus', 'idea_units']].dropna(subset=['stimulus'])

# Merge the combined data with the idea-units workbook so coverage comes only from the idea-units file.
df_merged = pd.merge(df, idea_units_df, on='stimulus', how='inner')

# Compute idea-unit coverage from the parsed unit list in the idea-units workbook.
df_merged['idea_unit_coverage'] = df_merged.apply(
    lambda row: idea_unit_coverage_from_excel(row['final transcription'], row['idea_units']),
    axis=1,
)

target = pd.to_numeric(df_merged['final score'], errors='coerce')
feature_frame = pd.DataFrame({
    'mer': df_merged.apply(lambda row: mer(clean_text(row['stimulus']), clean_text(row['final transcription'])), axis=1),
    'levenshtein_similarity': df_merged.apply(lambda row: levenshtein_similarity(row['stimulus'], row['final transcription']), axis=1),
    'meteor': df_merged.apply(lambda row: meteor_safe(row['stimulus'], row['final transcription']), axis=1),
    'idea_unit_coverage': pd.to_numeric(df_merged['idea_unit_coverage'], errors='coerce'),
    'sentence_length': df_merged['final transcription'].apply(sentence_length),
    'syllable_coverage': df_merged.apply(lambda row: syllable_coverage(row['stimulus'], row['final transcription']), axis=1),
})

model_df = pd.concat([feature_frame, target.rename('final score')], axis=1).dropna()
X = model_df[feature_frame.columns]
y = model_df['final score'].astype(int)

if y.nunique() < 2:
    raise ValueError('Need at least two score classes to fit multinomial logistic regression.')

min_class_count = int(y.value_counts().min())
if min_class_count < 2:
    raise ValueError('Each class needs at least two examples for stratified cross-validation.')

cv = StratifiedKFold(n_splits=min(5, min_class_count), shuffle=True, random_state=42)
from mord import LogisticAT
from itertools import combinations

# replace the pipeline and evaluate function
pipeline_ordinal = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticAT(alpha=1.0)),
])

def evaluate_ordinal(feature_names):
    preds = cross_val_predict(pipeline_ordinal, X[list(feature_names)], y, cv=cv)
    return {
        'features': ', '.join(feature_names),
        'n_features': len(feature_names),
        'accuracy': accuracy_score(y, preds),
        'weighted_f1': f1_score(y, preds, average='weighted', zero_division=0),
        'quadratic_kappa': cohen_kappa_score(y, preds, weights='quadratic'),
        'mae': np.mean(np.abs(preds - y)),
    }

# ablation: full set + leave-one-out
full_features = list(X.columns)
results = [evaluate_ordinal(full_features)]

for feat in full_features:
    reduced = [f for f in full_features if f != feat]
    results.append(evaluate_ordinal(reduced))

results_df = pd.DataFrame(results).sort_values('quadratic_kappa', ascending=False)
print('Ordinal Logistic Regression — Leave-One-Out Ablation (Cross-Validated)')
print(results_df.to_string(index=False))

Ordinal Logistic Regression — Leave-One-Out Ablation (Cross-Validated)
                                                                                   features  n_features  accuracy  weighted_f1  quadratic_kappa      mae
                    mer, levenshtein_similarity, meteor, sentence_length, syllable_coverage           5  0.669948     0.669724         0.888917 0.340415
mer, levenshtein_similarity, meteor, idea_unit_coverage, sentence_length, syllable_coverage           6  0.667876     0.667679         0.886668 0.344041
     levenshtein_similarity, meteor, idea_unit_coverage, sentence_length, syllable_coverage           5  0.665803     0.665682         0.885950 0.346114
        mer, levenshtein_similarity, idea_unit_coverage, sentence_length, syllable_coverage           5  0.654404     0.654583         0.879396 0.361140
                        mer, meteor, idea_unit_coverage, sentence_length, syllable_coverage           5  0.645596     0.645963         0.876374 0.368912
           

## SVM + TF-IDF vs Engineered Features (Scientific Comparison)

This section asks a focused research question:

> Do lexical text representations (TF-IDF) outperform engineered linguistic features for predicting final score?

We compare three cross-validated models on the same rows and folds:

1. **Engineered + LogisticRegression** (interpretable baseline)
2. **Engineered + LinearSVC** (same feature family, different classifier)
3. **TF-IDF + LinearSVC** (lexical representation)

Metrics reported include weighted F1, quadratic weighted kappa, and MAE (ordinal-sensitive).

## Engineered vs lexical comparison

This experiment asks whether hand-crafted linguistic features are more useful than a pure lexical representation.

Models are compared on the same rows and the same stratified folds:
- engineered features with logistic regression
- engineered features with a standard linear SVM
- engineered features with an ordinal linear SVM
- TF-IDF with a linear SVM

This block is mainly a classifier-family comparison, not a feature-ablation study.

In [13]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


class OrdinalLinearSVM(BaseEstimator, ClassifierMixin):
    def __init__(self, C=1.0, random_state=42):
        self.C = C
        self.random_state = random_state

    def fit(self, X, y):
        y_series = pd.Series(y)
        self.classes_ = np.sort(y_series.unique())
        if len(self.classes_) < 2:
            raise ValueError('OrdinalLinearSVM requires at least two classes.')

        self.thresholds_ = self.classes_[:-1]
        self.models_ = []
        self.constants_ = []

        for threshold in self.thresholds_:
            y_bin = (y_series > threshold).astype(int)
            if y_bin.nunique() < 2:
                self.models_.append(None)
                self.constants_.append(int(y_bin.iloc[0]))
                continue

            clf = LinearSVC(C=self.C, random_state=self.random_state)
            clf.fit(X, y_bin)
            self.models_.append(clf)
            self.constants_.append(None)

        return self

    def predict(self, X):
        threshold_votes = []
        for model, const in zip(self.models_, self.constants_):
            if model is None:
                threshold_votes.append(np.full(X.shape[0], const, dtype=int))
            else:
                threshold_votes.append(model.predict(X).astype(int))

        vote_matrix = np.vstack(threshold_votes).T
        class_positions = vote_matrix.sum(axis=1)
        return self.classes_[class_positions]


# Use the same rows across all models for an apples-to-apples comparison.
aligned_index = model_df.index
text_series = df_merged.loc[aligned_index, 'final transcription'].astype(str)
y_aligned = model_df['final score'].astype(int)
X_engineered = model_df[feature_frame.columns]

min_class_count = int(y_aligned.value_counts().min())
if min_class_count < 2:
    raise ValueError('Each class needs at least two examples for stratified cross-validation.')

cv_svm = StratifiedKFold(n_splits=min(5, min_class_count), shuffle=True, random_state=42)

def cross_val_preds(estimator, X_data, y_data, cv_obj):
    preds = pd.Series(index=y_data.index, dtype=float)
    for train_idx, test_idx in cv_obj.split(X_data, y_data):
        if hasattr(X_data, 'iloc'):
            X_train_fold = X_data.iloc[train_idx]
            X_test_fold = X_data.iloc[test_idx]
        else:
            X_train_fold = X_data[train_idx]
            X_test_fold = X_data[test_idx]

        y_train_fold = y_data.iloc[train_idx]
        model = clone(estimator)
        model.fit(X_train_fold, y_train_fold)
        preds.iloc[test_idx] = model.predict(X_test_fold)

    return preds.astype(int)

engineered_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='lbfgs', max_iter=3000, random_state=42)),
])

engineered_linsvm = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(C=1.0, random_state=42)),
])

engineered_ordinal_linsvm = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', OrdinalLinearSVM(C=1.0, random_state=42)),
])

tfidf_linsvm = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=500, lowercase=True, ngram_range=(1, 2))),
    ('clf', LinearSVC(C=1.0, random_state=42)),
])

model_specs = [
    ('Engineered + LogisticRegression', engineered_lr, X_engineered),
    ('Engineered + LinearSVC', engineered_linsvm, X_engineered),
    ('Engineered + Ordinal LinearSVM', engineered_ordinal_linsvm, X_engineered),
    ('TF-IDF + LinearSVC', tfidf_linsvm, text_series),
]

comparison_rows = []
for name, estimator, X_data in model_specs:
    preds = cross_val_preds(estimator, X_data, y_aligned, cv_svm)
    comparison_rows.append({
        'model': name,
        'accuracy': accuracy_score(y_aligned, preds),
        'weighted_f1': f1_score(y_aligned, preds, average='weighted'),
        'quadratic_kappa': cohen_kappa_score(y_aligned, preds, weights='quadratic'),
        'mae': mean_absolute_error(y_aligned, preds),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ['weighted_f1', 'quadratic_kappa'],
    ascending=False,
)

print('Lexical vs Engineered Feature Comparison (Cross-Validated)')
print(comparison_df.to_string(index=False))

Lexical vs Engineered Feature Comparison (Cross-Validated)
                          model  accuracy  weighted_f1  quadratic_kappa      mae
 Engineered + Ordinal LinearSVM  0.696373     0.695819         0.897619 0.313990
Engineered + LogisticRegression  0.688601     0.685411         0.893356 0.324352
         Engineered + LinearSVC  0.632642     0.588998         0.864975 0.395337
             TF-IDF + LinearSVC  0.513472     0.511046         0.749185 0.594301


In [14]:
print(feature_frame.columns.tolist())

['mer', 'levenshtein_similarity', 'meteor', 'idea_unit_coverage', 'sentence_length', 'syllable_coverage']


# Levenshtein Similarity ONLY

notes:
- sentence-length is less important than syllable count coverage statistically
- we CAN change syllable coverage to exact syllable matching as well, but this would fully violate semantic matching. 
- we can't switch to only levenshtein.
- sentence length is a fair metric, but 1.


# lowest MAE: Levenshtein + syllable + sentence length + meteor                          levenshtein_similarity, syllable_coverage, sentence_length, meteor           4  0.672021     0.672172         0.889258 0.338342
- removing sent and syll had the same effect


In [15]:
# Levenshtein-only vs ablations
lev_only = ['levenshtein_similarity']
full_features = list(X.columns)
without_lev = [feature for feature in full_features if feature != 'levenshtein_similarity']
without_idea = [feature for feature in full_features if feature != 'idea_unit_coverage']
no_syllable = [feature for feature in full_features if feature != 'syllable_coverage' and feature != 'idea_unit_coverage']
only_lev_syll_sentence = ['levenshtein_similarity', 'syllable_coverage', 'sentence_length']
lev_syll_sentence_meteor = ['levenshtein_similarity', 'syllable_coverage', 'sentence_length', 'meteor']
lev_syll_meteor = ['levenshtein_similarity', 'syllable_coverage', 'meteor']
lev_sentence_meteor = ['levenshtein_similarity', 'sentence_length', 'meteor']
lev_meteor = ['levenshtein_similarity', 'meteor']


comparison_specs = [
    ('Levenshtein only', lev_only),
    ('Full feature model', full_features),
    ('Without Levenshtein', without_lev),
    ('Without idea-unit coverage', without_idea),
    ('Without syllable and idea-unit coverage', no_syllable),
    ('Levenshtein + syllable + sentence length', only_lev_syll_sentence),
    ('Levenshtein + syllable + sentence length + meteor', lev_syll_sentence_meteor),
    ('Levenshtein + syllable + meteor', lev_syll_meteor),
    ('Levenshtein + sentence length + meteor', lev_sentence_meteor),    
    ('Levenshtein + meteor', lev_meteor)
]

comparison_rows = []
for name, feature_names in comparison_specs:
    predictions = cross_val_predict(pipeline_ordinal, X[feature_names], y, cv=cv)
    comparison_rows.append({
        'model': name,
        'features': ', '.join(feature_names),
        'n_features': len(feature_names),
        'accuracy': accuracy_score(y, predictions),
        'weighted_f1': f1_score(y, predictions, average='weighted', zero_division=0),
        'quadratic_kappa': cohen_kappa_score(y, predictions, weights='quadratic'),
        'mae': np.mean(np.abs(predictions - y)),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ['weighted_f1', 'quadratic_kappa'],
    ascending=False,
)

print('Levenshtein-only comparison (Cross-Validated)')
print(comparison_df.to_string(index=False))

Levenshtein-only comparison (Cross-Validated)
                                            model                                                                                    features  n_features  accuracy  weighted_f1  quadratic_kappa      mae
Levenshtein + syllable + sentence length + meteor                          levenshtein_similarity, syllable_coverage, sentence_length, meteor           4  0.672021     0.672172         0.889258 0.338342
                       Without idea-unit coverage                     mer, levenshtein_similarity, meteor, sentence_length, syllable_coverage           5  0.669948     0.669724         0.888917 0.340415
                               Full feature model mer, levenshtein_similarity, meteor, idea_unit_coverage, sentence_length, syllable_coverage           6  0.667876     0.667679         0.886668 0.344041
         Levenshtein + syllable + sentence length                                  levenshtein_similarity, syllable_coverage, sentence_length 

BETO

In [16]:
# BETO baseline vs final score
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.decomposition import PCA

beto_model_name = 'dccuchile/bert-base-spanish-wwm-cased'
beto_tokenizer = AutoTokenizer.from_pretrained(beto_model_name)
beto_model = AutoModel.from_pretrained(beto_model_name)
beto_model.eval()


# Build a single combined BETO input so the model always sees both sequences.
df_merged['beto_input'] = df_merged.apply(
    lambda row: f"{clean_text(row['stimulus'])} [SEP] {clean_text(row['final transcription'])}",
    axis=1,
)

beto_text = df_merged.loc[model_df.index, 'beto_input'].astype(str)


def beto_embed(text, max_length=128):
    text = clean_text(text)
    if text is None:
        return np.full(768, np.nan)

    inputs = beto_tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding=True,
    )
    with torch.no_grad():
        outputs = beto_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()


beto_matrix = np.vstack([beto_embed(text) for text in beto_text])
beto_matrix = pd.DataFrame(beto_matrix).replace([np.inf, -np.inf], np.nan)

# Reduce dimensionality to keep the CV model stable on a small sample.
n_components = min(50, beto_matrix.shape[0] - 1, beto_matrix.shape[1])
if n_components < 2:
    raise ValueError('Not enough rows to evaluate BETO embeddings.')

beto_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('pca', PCA(n_components=n_components, random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='lbfgs', max_iter=3000, random_state=42)),
])

beto_predictions = cross_val_preds(beto_pipeline, beto_matrix.fillna(0), y_aligned, cv_svm)
beto_results = pd.DataFrame([
    {
        'model': 'BETO + LogisticRegression',
        'accuracy': accuracy_score(y_aligned, beto_predictions),
        'weighted_f1': f1_score(y_aligned, beto_predictions, average='weighted', zero_division=0),
        'quadratic_kappa': cohen_kappa_score(y_aligned, beto_predictions, weights='quadratic'),
        'mae': mean_absolute_error(y_aligned, beto_predictions),
    }
])

print('BETO Baseline (Cross-Validated)')
print(beto_results.to_string(index=False))


Loading weights: 100%|██████████| 197/197 [00:01<00:00, 173.03it/s, Materializing param=encoder.layer.11.output.dense.weight]              
BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

BETO Baseline (Cross-Validated)
                    model  accuracy  weighted_f1  quadratic_kappa      mae
BETO + LogisticRegression       0.6     0.598233         0.837774 0.443523


## BETO baseline

This block evaluates BETO as a frozen embedding model.

The stimulus and transcription are concatenated with a [SEP] token so the transformer sees both sequences at once. The resulting embedding is then reduced with PCA and passed to logistic regression.

This is a strong baseline for asking whether a general Spanish transformer adds signal beyond the engineered features.

In [17]:
# BETO mean-pooling + ordinal classifier

def beto_mean_pool_embeddings(texts, max_length=128, batch_size=16):
    pooled_batches = []
    text_list = [str(text) for text in texts]

    for start in range(0, len(text_list), batch_size):
        batch_texts = text_list[start:start + batch_size]
        inputs = beto_tokenizer(
            batch_texts,
            return_tensors='pt',
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        with torch.no_grad():
            outputs = beto_model(**inputs)

        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs['attention_mask'].unsqueeze(-1).type_as(token_embeddings)
        pooled = (token_embeddings * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1e-9)
        pooled_batches.append(pooled.cpu().numpy())

    return np.vstack(pooled_batches)

beto_mean_matrix = beto_mean_pool_embeddings(beto_text, max_length=128, batch_size=16)
beto_mean_matrix = pd.DataFrame(beto_mean_matrix).replace([np.inf, -np.inf], np.nan)

n_components_mean = min(50, beto_mean_matrix.shape[0] - 1, beto_mean_matrix.shape[1])
if n_components_mean < 2:
    raise ValueError('Not enough rows to evaluate BETO mean-pooled embeddings.')

beto_mean_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('pca', PCA(n_components=n_components_mean, random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LogisticAT(alpha=1.0)),
])

beto_mean_predictions = cross_val_preds(beto_mean_pipeline, beto_mean_matrix.fillna(0), y_aligned, cv_svm)
beto_mean_results = pd.DataFrame([
    {
        'model': 'BETO mean pooling + LogisticAT',
        'accuracy': accuracy_score(y_aligned, beto_mean_predictions),
        'weighted_f1': f1_score(y_aligned, beto_mean_predictions, average='weighted', zero_division=0),
        'quadratic_kappa': cohen_kappa_score(y_aligned, beto_mean_predictions, weights='quadratic'),
        'mae': mean_absolute_error(y_aligned, beto_mean_predictions),
    }
])

beto_comparison = pd.concat([
    beto_results,
    beto_mean_results,
], ignore_index=True).sort_values(['quadratic_kappa', 'weighted_f1'], ascending=False)

print('BETO Comparison (Cross-Validated)')
print(beto_comparison.to_string(index=False))


BETO Comparison (Cross-Validated)
                         model  accuracy  weighted_f1  quadratic_kappa      mae
BETO mean pooling + LogisticAT  0.601554     0.601657         0.855374 0.419171
     BETO + LogisticRegression  0.600000     0.598233         0.837774 0.443523


In [18]:
# Ordinal Linear SVM with reduced engineered feature set
ordinal_svm_features = [
    'levenshtein_similarity',
    'syllable_coverage',
    'sentence_length',
    'meteor',
]

ordinal_svm_subset = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', OrdinalLinearSVM(C=1.0, random_state=42)),
])

subset_predictions = cross_val_preds(ordinal_svm_subset, X_engineered[ordinal_svm_features], y_aligned, cv_svm)
subset_results = pd.DataFrame([
    {
        'model': 'Ordinal LinearSVM (selected features)',
        'features': ', '.join(ordinal_svm_features),
        'n_features': len(ordinal_svm_features),
        'accuracy': accuracy_score(y_aligned, subset_predictions),
        'weighted_f1': f1_score(y_aligned, subset_predictions, average='weighted', zero_division=0),
        'quadratic_kappa': cohen_kappa_score(y_aligned, subset_predictions, weights='quadratic'),
        'mae': mean_absolute_error(y_aligned, subset_predictions),
    }
])

print('Ordinal Linear SVM (Selected Features, Cross-Validated)')
print(subset_results.to_string(index=False))

Ordinal Linear SVM (Selected Features, Cross-Validated)
                                model                                                           features  n_features  accuracy  weighted_f1  quadratic_kappa      mae
Ordinal LinearSVM (selected features) levenshtein_similarity, syllable_coverage, sentence_length, meteor           4  0.689119     0.688629         0.892832 0.322798


## Selected engineered ordinal SVM

This block isolates a smaller engineered feature set and evaluates it with the custom ordinal linear SVM.

The goal is to see whether the simpler feature bundle gives a better bias-variance tradeoff than the full engineered set, especially on ordinal metrics such as quadratic weighted kappa and mean absolute error.

In [19]:
# Engineered + Ordinal LinearSVM on the four selected feature sets
engineered_ordinal_sets = [
    ('Set 1: levenshtein + syllable + sentence length + meteor', [
        'levenshtein_similarity',
        'syllable_coverage',
        'sentence_length',
        'meteor',
    ]),
    ('Set 2: mer + levenshtein + meteor + sentence length + syllable', [
        'mer',
        'levenshtein_similarity',
        'meteor',
        'sentence_length',
        'syllable_coverage',
    ]),
    ('Set 3: mer + levenshtein + meteor + idea-unit + sentence length + syllable', [
        'mer',
        'levenshtein_similarity',
        'meteor',
        'idea_unit_coverage',
        'sentence_length',
        'syllable_coverage',
    ]),
    ('Set 4: levenshtein + syllable + sentence length', [
        'levenshtein_similarity',
        'syllable_coverage',
        'sentence_length',
    ]),
]

engineered_ordinal_rows = []
for set_name, feature_names in engineered_ordinal_sets:
    predictions = cross_val_preds(ordinal_svm_subset, X_engineered[feature_names], y_aligned, cv_svm)
    engineered_ordinal_rows.append({
        'model': set_name,
        'features': ', '.join(feature_names),
        'n_features': len(feature_names),
        'accuracy': accuracy_score(y_aligned, predictions),
        'weighted_f1': f1_score(y_aligned, predictions, average='weighted', zero_division=0),
        'quadratic_kappa': cohen_kappa_score(y_aligned, predictions, weights='quadratic'),
        'mae': mean_absolute_error(y_aligned, predictions),
    })

engineered_ordinal_df = pd.DataFrame(engineered_ordinal_rows).sort_values(
    ['quadratic_kappa', 'weighted_f1'],
    ascending=False,
)

print('Engineered + Ordinal LinearSVM on Selected Feature Sets (Cross-Validated)')
print(engineered_ordinal_df.to_string(index=False))

Engineered + Ordinal LinearSVM on Selected Feature Sets (Cross-Validated)
                                                                     model                                                                                    features  n_features  accuracy  weighted_f1  quadratic_kappa      mae
Set 3: mer + levenshtein + meteor + idea-unit + sentence length + syllable mer, levenshtein_similarity, meteor, idea_unit_coverage, sentence_length, syllable_coverage           6  0.696373     0.695819         0.897619 0.313990
            Set 2: mer + levenshtein + meteor + sentence length + syllable                     mer, levenshtein_similarity, meteor, sentence_length, syllable_coverage           5  0.693782     0.693096         0.895137 0.317617
                  Set 1: levenshtein + syllable + sentence length + meteor                          levenshtein_similarity, syllable_coverage, sentence_length, meteor           4  0.689119     0.688629         0.892832 0.322798
              

## Four feature bundles

These four runs compare the same ordinal SVM wrapper across manually chosen feature bundles.

The point is to see whether adding MER or idea-unit coverage helps once the lexical overlap and length-based signals are already present.

This is the most direct feature-set comparison in the notebook.

## Summary of findings and usage

The notebook is organized around three practical questions:
1. Which engineered features are most predictive under an ordinal model?
2. Do lexical models like TF-IDF or BETO outperform hand-crafted features?
3. Which model is most defensible to ship to other researchers working on Spanish EIT?

The most reproducible pieces are the engineered-feature experiments, because they use explicit rules, fixed CV splits, and no external checkpoint dependencies. BETO is useful as a higher-capacity comparison, but it is less portable because it depends on a pretrained checkpoint, tokenization details, and more runtime moving parts.

For a final report, the safest interpretation is usually:
- MER as a simple baseline
- engineered ordinal models as the main transparent benchmark
- BETO as the neural comparison model

In [20]:
# --- Hybrid experiments: BETO cosine-sim and appended embeddings
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np
import torch

# locate transcription column
trans_cols = [c for c in df_merged.columns if 'transcript' in c.lower() or 'transcription' in c.lower()]
if not trans_cols:
    raise RuntimeError('No transcription-like column found in df_merged')
trans_col = trans_cols[0]

# helper to embed texts (mean pooling)
def embed_texts(texts, tokenizer=beto_tokenizer, model=beto_model, batch_size=32):
    model.eval()
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = list(texts[i:i+batch_size])
            enc = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
            for k,v in enc.items():
                if isinstance(v, torch.Tensor):
                    enc[k] = v.to(next(model.parameters()).device)
            out = model(**enc)
            last = out.last_hidden_state
            mask = enc['attention_mask'].unsqueeze(-1)
            summed = (last * mask).sum(1)
            counts = mask.sum(1)
            emb = (summed / counts).cpu().numpy()
            embs.append(emb)
    return np.vstack(embs)

# compute embeddings for stimulus and transcription (aligned indices)
stim_texts = df_merged.loc[aligned_index, stimulus_col].fillna('').astype(str)
trans_texts = df_merged.loc[aligned_index, trans_col].fillna('').astype(str)
print('Embedding', len(stim_texts), 'stimulus and', len(trans_texts), 'transcription texts')
stim_emb = embed_texts(stim_texts)
trans_emb = embed_texts(trans_texts)

# cosine similarity (row-wise)
cos_sim = np.array([cosine_similarity(stim_emb[i:i+1], trans_emb[i:i+1])[0,0] for i in range(stim_emb.shape[0])])

# build feature frame copy and replace meteor (if present) or add beto_cosine_sim
X_base = feature_frame.loc[aligned_index].copy()
if 'meteor' in X_base.columns:
    X_base['meteor'] = cos_sim
else:
    X_base['beto_cosine_sim'] = cos_sim

# run cross-validated predictions with engineered ordinal linear SVM
preds_cos = cross_val_predict(engineered_ordinal_linsvm, X_base, y_aligned, cv=cv_svm)

res_cos = {
    'accuracy': accuracy_score(y_aligned, preds_cos),
    'weighted_f1': f1_score(y_aligned, preds_cos, average='weighted'),
    'qwk': cohen_kappa_score(y_aligned, preds_cos, weights='quadratic'),
    'mae': mean_absolute_error(y_aligned, preds_cos)
}
print('\nBETO cosine-sim + engineered model results:')
print(res_cos)

# --- Hybrid: append BETO semantic embeddings to engineered features (Leak-Free CV Pipeline)
# Create a DataFrame combining base engineered features with raw 768-dim BETO embeddings
beto_raw_df = pd.DataFrame(
    stim_emb, 
    index=aligned_index, 
    columns=[f'beto_raw_{i}' for i in range(stim_emb.shape[1])]
)
X_app_raw = pd.concat([X_base, beto_raw_df], axis=1)

engineered_cols = list(X_base.columns)
beto_cols = list(beto_raw_df.columns)
n_comp = min(50, len(aligned_index) - 1, stim_emb.shape[1])

# Construct ColumnTransformer so PCA is fit ONLY on training folds inside each CV split
preprocessor = ColumnTransformer(
    transformers=[
        ('eng', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), engineered_cols),
        ('beto', Pipeline([
            ('pca', PCA(n_components=n_comp, random_state=42)),
            ('scaler', StandardScaler())
        ]), beto_cols)
    ]
)

pipeline_app_clean = Pipeline([
    ('prep', preprocessor),
    ('clf', OrdinalLinearSVM(C=1.0, random_state=42))
])

preds_app = cross_val_preds(pipeline_app_clean, X_app_raw, y_aligned, cv_svm)
res_app = {
    'accuracy': accuracy_score(y_aligned, preds_app),
    'weighted_f1': f1_score(y_aligned, preds_app, average='weighted'),
    'qwk': cohen_kappa_score(y_aligned, preds_app, weights='quadratic'),
    'mae': mean_absolute_error(y_aligned, preds_app)
}
print('\nEngineered + BETO-embeddings (appended, leak-free CV) results:')
print(res_app)

# expose results to notebook namespace
beto_hybrid_cos_results = res_cos
beto_hybrid_appended_results = res_app
beto_cosine_sim = cos_sim


Embedding 1930 stimulus and 1930 transcription texts

BETO cosine-sim + engineered model results:
{'accuracy': 0.6922279792746114, 'weighted_f1': 0.6913049511184908, 'qwk': 0.896677726191289, 'mae': 0.3176165803108808}

Engineered + BETO-embeddings (appended, leak-free CV) results:
{'accuracy': 0.6813471502590673, 'weighted_f1': 0.6811546323946831, 'qwk': 0.894832866675262, 'mae': 0.32746113989637304}


## Experiment A: Cross-Encoder Joint BETO Representation ([CLS] & Mean-Pooling + Ordinal LogisticAT)

In this experiment, stimulus and transcription are fed jointly into BETO as a single sequence: `[CLS] stimulus [SEP] transcription [SEP]`.
This allows full cross-attention across tokens. We evaluate both `[CLS]` token pooling and mean-pooling with an ordinal classifier (`LogisticAT`) using leak-free CV pipelines.

In [21]:
# Experiment A: Cross-Encoder Joint BETO Representation ([CLS] & Mean Pooling + Ordinal LogisticAT)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from mord import LogisticAT
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, mean_absolute_error
import pandas as pd
import numpy as np

# 1. Joint Cross-Encoder input text
df_merged['beto_input'] = df_merged.apply(
    lambda row: f"{clean_text(row['stimulus'])} [SEP] {clean_text(row['final transcription'])}",
    axis=1,
)
beto_text = df_merged.loc[model_df.index, 'beto_input'].astype(str)

# 2. Extract joint [CLS] embeddings
def get_beto_cls_matrix(texts):
    cls_list = []
    for text in texts:
        inputs = beto_tokenizer(str(text), return_tensors='pt', truncation=True, max_length=128, padding=True)
        with torch.no_grad():
            outputs = beto_model(**inputs)
        cls_list.append(outputs.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy())
    return np.vstack(cls_list)

beto_cls_matrix = pd.DataFrame(get_beto_cls_matrix(beto_text), index=model_df.index)

# Leak-free CV pipeline for Cross-Encoder [CLS] + Ordinal LogisticAT
n_comp_beto = min(50, beto_cls_matrix.shape[0] - 1, beto_cls_matrix.shape[1])
beto_cls_ordinal_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('pca', PCA(n_components=n_comp_beto, random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', LogisticAT(alpha=1.0)),
])

preds_cls_ordinal = cross_val_preds(beto_cls_ordinal_pipe, beto_cls_matrix.fillna(0), y_aligned, cv_svm)
res_cls_ordinal = {
    'model': 'BETO Cross-Encoder [CLS] + LogisticAT',
    'accuracy': accuracy_score(y_aligned, preds_cls_ordinal),
    'weighted_f1': f1_score(y_aligned, preds_cls_ordinal, average='weighted', zero_division=0),
    'quadratic_kappa': cohen_kappa_score(y_aligned, preds_cls_ordinal, weights='quadratic'),
    'mae': mean_absolute_error(y_aligned, preds_cls_ordinal)
}
print("--- Cross-Encoder [CLS] + LogisticAT Results ---")
print(res_cls_ordinal)


--- Cross-Encoder [CLS] + LogisticAT Results ---
{'model': 'BETO Cross-Encoder [CLS] + LogisticAT', 'accuracy': 0.5590673575129533, 'weighted_f1': 0.5611786685172155, 'quadratic_kappa': 0.826481683308414, 'mae': 0.4761658031088083}


## Final Summary Table: All Model Tiers (TF-IDF vs Pure BETO vs Engineered vs Hybrid)

This summary table presents the complete comparison across all baseline, pure neural, engineered, and hybrid models under leak-free cross-validation.

### BERTScore notes

- We compute per-row BERTScore F1 using `bert-score` as a semantic similarity feature.
- Preferred model: `bert-base-multilingual-cased` for robust Spanish coverage; the cell will install `bert-score` if missing.
- The BERTScore feature replaces `meteor` in the engineered features when present.
- Reproducibility: use `lang='es'`, `rescale_with_baseline=False`, and `batch_size=32`.

In [25]:
# --- Compute BERTScore per-row and evaluate engineered model using it
# Uses bert-score package; falls back to installing if needed
try:
    from bert_score import score as bert_score_score
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'bert-score'])
    from bert_score import score as bert_score_score

# prepare texts (reference=stimulus, candidate=transcription)
trans_cols = [c for c in df_merged.columns if 'transcript' in c.lower() or 'transcription' in c.lower()]
if not trans_cols:
    raise RuntimeError('No transcription-like column found in df_merged')
trans_col = trans_cols[0]
refs = df_merged.loc[aligned_index, stimulus_col].fillna('').astype(str).tolist()
cands = df_merged.loc[aligned_index, trans_col].fillna('').astype(str).tolist()

# compute BERTScore F1 per sample
# use a bert-score-supported multilingual model for robust Spanish support
bs_model = 'bert-base-multilingual-cased'
print('Computing BERTScore for', len(cands), 'pairs using model', bs_model)
P, R, F1 = bert_score_score(cands, refs, model_type=bs_model, lang='es', rescale_with_baseline=False, verbose=False, batch_size=32)
# convert to numpy
import numpy as np
if hasattr(F1, 'cpu'):
    berts = F1.cpu().numpy()
elif isinstance(F1, np.ndarray):
    berts = F1
else:
    berts = np.array(F1)

# add to feature frame (replace meteor if present)
X_bs = feature_frame.loc[aligned_index].copy()
if 'meteor' in X_bs.columns:
    X_bs['meteor'] = berts
else:
    X_bs['bertscore_f1'] = berts

# evaluate engineered ordinal linear SVM
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score
preds_bs = cross_val_predict(engineered_ordinal_linsvm, X_bs, y_aligned, cv=cv_svm)
res_bs = {
    'accuracy': accuracy_score(y_aligned, preds_bs),
    'weighted_f1': f1_score(y_aligned, preds_bs, average='weighted'),
    'qwk': cohen_kappa_score(y_aligned, preds_bs, weights='quadratic'),
    'mae': mean_absolute_error(y_aligned, preds_bs)
}
print('\nEngineered + BERTScore (F1) results:')
print(res_bs)

# expose for later use
bertscore_per_row = berts
bertscore_engineered_results = res_bs

Computing BERTScore for 1930 pairs using model bert-base-multilingual-cased


C:\Users\khann\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\khann\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100


Engineered + BERTScore (F1) results:
{'accuracy': 0.6860103626943005, 'weighted_f1': 0.6839712840197876, 'qwk': 0.8946867373789489, 'mae': 0.3243523316062176}


## Ordinal decomposition (Threshold ensemble)

This cell implements an ordinal decomposition where independent binary classifiers
are trained for each threshold (e.g., >=1, >=2, >=3, >=4). Final predictions are
produced by summing positive threshold votes and mapping the vote count to the
original ordinal labels. This approach gives per-threshold calibration signals
and can improve boundary behavior compared to a single multiclass classifier.

The produced variables are `threshold_ensemble_preds`, `threshold_ensemble_votes`,
and `threshold_ensemble_results` which are included in the final summary table
when present. Cross-validation is leak-free: each binary classifier is trained
inside the same `cv_svm` folds used elsewhere in the notebook.

In [28]:
# --- Threshold ensemble (ordinal decomposition)
# Train a binary classifier for each threshold (>=1, >=2, >=3, >=4 or inferred thresholds),
# then sum positive votes to produce final ordinal prediction.

from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score
import numpy as np
import pandas as pd

# Choose feature matrix to use (change if you want a different feature set)
X_feat = X_engineered.copy()   # or X_base / X_app_raw / X_app depending on experiment

# Determine thresholds: use explicit 1..4 if present, otherwise infer from classes
explicit_thresholds = [1, 2, 3, 4]
classes_sorted = np.sort(y_aligned.unique())
if all(t in classes_sorted for t in explicit_thresholds):
    thresholds = explicit_thresholds
else:
    thresholds = list(classes_sorted[:-1])  # infer ordinal boundaries

# Base binary estimator (preprocessing per-fold handled by cross_val_preds via cloning the pipeline)
binary_estimator = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(C=1.0, random_state=42))
])

# Collect cross-validated binary predictions for each threshold
votes = pd.DataFrame(index=y_aligned.index)
for t in thresholds:
    y_bin = (y_aligned >= t).astype(int)
    preds_t = cross_val_preds(binary_estimator, X_feat, y_bin, cv_svm)  # uses existing cross_val_preds
    votes[f'thresh_{t}'] = preds_t.astype(int)

# Sum votes -> position index into classes_sorted
vote_counts = votes.sum(axis=1).astype(int)  # ranges 0 .. (n_thresholds)
# Map vote count to actual class labels (classes_sorted length = n_thresholds+1)
preds_threshold_ensemble = pd.Series(classes_sorted[vote_counts], index=y_aligned.index)

# Metrics
res_threshold_ensemble = {
    'accuracy': accuracy_score(y_aligned, preds_threshold_ensemble),
    'weighted_f1': f1_score(y_aligned, preds_threshold_ensemble, average='weighted'),
    'qwk': cohen_kappa_score(y_aligned, preds_threshold_ensemble, weights='quadratic'),
    'mae': mean_absolute_error(y_aligned, preds_threshold_ensemble)
}

print('Threshold ensemble (ordinal decomposition) results:')
print(res_threshold_ensemble)

# Expose for later use / comparison
threshold_ensemble_preds = preds_threshold_ensemble
threshold_ensemble_votes = votes
threshold_ensemble_results = res_threshold_ensemble

Threshold ensemble (ordinal decomposition) results:
{'accuracy': 0.6958549222797927, 'weighted_f1': 0.6947633590629722, 'qwk': 0.8975367486070237, 'mae': 0.3145077720207254}


In [30]:
# FINAL SUMMARY: compile all evaluated models (placed as the last cell for reproducibility)
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score

final_rows = []

# baseline list (will be re-evaluated using existing cross_val_preds to ensure same CV splits)
summary_models = [
    ('TF-IDF + LinearSVC', tfidf_linsvm, text_series),
    ('Engineered + LogisticRegression', engineered_lr, X_engineered),
    ('Engineered + Ordinal LinearSVM (Selected 4)', ordinal_svm_subset, X_engineered[ordinal_svm_features]),
    ('Engineered + Ordinal LinearSVM (Full 6)', engineered_ordinal_linsvm, X_engineered),
]

for name, estimator, X_data in summary_models:
    try:
        preds = cross_val_preds(estimator, X_data, y_aligned, cv_svm)
        final_rows.append({
            'model': name,
            'accuracy': accuracy_score(y_aligned, preds),
            'weighted_f1': f1_score(y_aligned, preds, average='weighted', zero_division=0),
            'quadratic_kappa': cohen_kappa_score(y_aligned, preds, weights='quadratic'),
            'mae': mean_absolute_error(y_aligned, preds),
        })
    except Exception as e:
        final_rows.append({'model': name, 'accuracy': np.nan, 'weighted_f1': np.nan, 'quadratic_kappa': np.nan, 'mae': np.nan})

# helper to normalize and append result-like dicts
def _append_result_like(d, model_name=None):
    try:
        acc = d.get('accuracy', d.get('acc', np.nan))
        wf = d.get('weighted_f1', np.nan)
        qwk = d.get('quadratic_kappa', d.get('qwk', np.nan))
        mae = d.get('mae', np.nan)
        name = model_name if model_name is not None else d.get('model', 'unnamed')
    except Exception:
        try:
            acc = float(d['accuracy'])
            wf = float(d.get('weighted_f1', np.nan))
            qwk = float(d.get('quadratic_kappa', d.get('qwk', np.nan)))
            mae = float(d.get('mae', np.nan))
            name = model_name if model_name is not None else d.get('model', 'unnamed')
        except Exception:
            acc, wf, qwk, mae, name = (np.nan, np.nan, np.nan, np.nan, model_name or 'unnamed')
    final_rows.append({
        'model': name,
        'accuracy': acc,
        'weighted_f1': wf,
        'quadratic_kappa': qwk,
        'mae': mae,
    })

# Append previously computed result dicts if available in the namespace
if 'res_cls_ordinal' in globals():
    _append_result_like(res_cls_ordinal)

if 'beto_results' in globals():
    try:
        _append_result_like(beto_results.iloc[0].to_dict(), model_name=beto_results.iloc[0].get('model', 'BETO + LogisticRegression'))
    except Exception:
        pass

if 'beto_mean_results' in globals():
    try:
        _append_result_like(beto_mean_results.iloc[0].to_dict(), model_name=beto_mean_results.iloc[0].get('model', 'BETO mean pooling + LogisticAT'))
    except Exception:
        pass

if 'res_cos' in globals():
    _append_result_like(res_cos, model_name='Hybrid: BETO Cosine Sim + Engineered Ordinal SVM')

if 'res_app' in globals():
    _append_result_like(res_app, model_name='Hybrid: Appended BETO PCA + Engineered Ordinal SVM (Leak-Free)')

if 'bertscore_engineered_results' in globals():
    _append_result_like(bertscore_engineered_results, model_name='Engineered + BERTScore (F1)')

if 'threshold_ensemble_results' in globals():
    _append_result_like(threshold_ensemble_results, model_name='Threshold ensemble (ordinal decomposition)')

# produce DataFrame and sort by QWK (descending)
summary_df = pd.DataFrame(final_rows).sort_values('quadratic_kappa', ascending=False)
print("=== COMPLETE MODEL COMPARISON SUMMARY (FINAL) ===")
print(summary_df.to_string(index=False))

# persist a CSV for the reproducibility table (optional)
try:
    summary_df.to_csv('../models/summary_table.csv', index=False)
    print('Saved summary table to ../models/summary_table.csv')
except Exception:
    pass


=== COMPLETE MODEL COMPARISON SUMMARY (FINAL) ===
                                                         model  accuracy  weighted_f1  quadratic_kappa      mae
                       Engineered + Ordinal LinearSVM (Full 6)  0.696373     0.695819         0.897619 0.313990
                    Threshold ensemble (ordinal decomposition)  0.695855     0.694763         0.897537 0.314508
              Hybrid: BETO Cosine Sim + Engineered Ordinal SVM  0.692228     0.691305         0.896678 0.317617
Hybrid: Appended BETO PCA + Engineered Ordinal SVM (Leak-Free)  0.681347     0.681155         0.894833 0.327461
                                   Engineered + BERTScore (F1)  0.686010     0.683971         0.894687 0.324352
                               Engineered + LogisticRegression  0.688601     0.685411         0.893356 0.324352
                   Engineered + Ordinal LinearSVM (Selected 4)  0.689119     0.688629         0.892832 0.322798
                                BETO mean pooling + Lo